#Handling Missing Data in ETL Assignment

#SECTION A – THEORETICAL QUESTIONS

#Question.1-  What are the most common reasons for missing data in ETL pipelines?

Answer - Missing data in ETL (Extract, Transform, Load) pipelines usually comes from issues at the source, during data movement, or during transformation and loading. The most common causes are:

1. **Source System Issues**

1.Incomplete data entry: Users leave fields blank in forms or applications.

2.Application bugs: Source systems fail to capture or store certain values.

3.Optional fields: Business processes allow fields to remain empty.

4.Data retention policies: Historical data is deleted or archived before extraction.

Example: A customer record exists, but the phone number field is empty because it was optional during registration.

2. **Extraction Problems**

1.Failed API calls: Network issues, rate limits, or authentication failures prevent retrieval of some records.

2.Incremental load errors: Incorrect watermark/timestamp logic skips records.

3.Query filtering mistakes: Extraction queries unintentionally exclude data.

4.Source outages: Systems are unavailable during extraction windows.

Example: An ETL job pulls only records updated after 2025-06-01 00:00:00, but timezone issues cause some records to be skipped.

3. **Transformation Errors**

1.Data type mismatches: Values fail conversion and are discarded.

2.Validation rules: Records are rejected because they don't meet quality standards.

3.Join failures: Missing lookup keys result in null values.

4.Parsing errors: Malformed JSON, XML, CSV, or log files cause fields to be lost.

Example: Customer records are joined to a country reference table, but unmatched country codes produce null country names.

4. **Loading Problems**

1.Constraint violations: Records fail due to primary key, foreign key, or uniqueness constraints.

2.Batch failures: Partial loads occur when jobs crash midway.

3.Truncation: Values exceeding field length limits are dropped or shortened.

4.Rejected records: Error rows are sent to quarantine tables and never reprocessed.

Example: A database column allows 50 characters, but incoming values contain 100 characters and are rejected.

5. **Data Integration Issues**

1.Schema changes: New columns are added in the source but not mapped in the ETL pipeline.

2.Field mapping errors: Source fields are mapped incorrectly or omitted.

3.Multiple source conflicts: Different systems provide inconsistent information.

Example: The source system renames customer_email to email_address, but the ETL mapping isn't updated.

6. **Timing and Synchronization Issues**

1.Late-arriving data: Data arrives after the ETL job has already run.

2.Race conditions: Data is updated while extraction is in progress.

3.Event ordering problems: Related records arrive in the wrong sequence.

Example: An order is loaded before the corresponding customer record, causing foreign-key-related gaps.

7. **Data Quality Problems**

1.Duplicate removal logic accidentally removes valid records.

2.Null propagation from upstream systems.

3.Corrupted files or malformed records.

4.Manual corrections that introduce inconsistencies.

#Question.2- Why is blindly deleting rows with missing values considered a bad practice in ETL?

Answer - Blindly deleting rows with missing values is generally considered a bad practice because it can reduce data quality, introduce bias, and hide underlying problems in the ETL pipeline. Whether a row should be removed depends on why the data is missing and how the data will be used.

Here are the main reasons:

1. **Loss of Valuable Information**
A row may have only one missing field while the remaining columns contain useful data.

2. **Introduction of Bias**
Missing values are often not random. Removing all incomplete records can distort the dataset.

For example:
Older customers may be less likely to provide email addresses.
Certain regions may have incomplete demographic information.

3. **Reduced Dataset Size**

Large-scale deletion can significantly reduce the amount of available data, especially if many records contain at least one missing value.

4. **Hides Upstream Data Quality Issues**
Missing values often indicate problems earlier in the pipeline, such as:

Failed API responses

Incorrect joins

Schema changes

Data entry issues

Parsing errors

5. **Different Columns Have Different Importance**
Not every missing value has the same impact.

For example:

Missing Customer ID may make a record unusable.
Missing Middle Name may have little or no impact.

6. **Better Alternatives Exist**

Depending on the context, you can often preserve more information by:

Imputing missing values (for example, using the mean, median, mode, or model-based methods where appropriate).

Filling with sensible defaults (such as "Unknown" or "Not Provided" when appropriate).

#Question.3 -Explain the difference between:

Listwise deletion

Column deletion

Also mention one scenario where each is appropriate?

Answer - **Listwise Deletion**

1.Removes entire rows (records) that contain one or more missing values.

2.Used when only a small percentage of rows have missing data.

3.Helps ensure that all remaining records are complete.

Appropriate scenario:

A sales dataset has only 2% of records with missing values, and the analysis requires complete customer information. Removing those few incomplete rows has minimal impact on the results.

**Column Deletion**

1.Removes entire columns (features) that contain many missing values.

2.Used when a column has a high percentage of missing data or is not important for the analysis.
3.Keeps all records while eliminating an unreliable feature.

Appropriate scenario:

A customer survey includes an optional question that 95% of respondents skipped. Since the column contains very little useful data, it is removed from the dataset.

#Question.4 - Why is median imputation preferred over mean imputation for skewed data such as income?

Answer - Median imputation is preferred over mean imputation for skewed data (such as income) because the median is not affected by extreme values (outliers), while the mean is.

Reasons:

1.Median is resistant to outliers, so it better represents the typical value in a skewed distribution.

2.Mean is pulled toward very high or very low values, making it less representative of most observations.

3.Income data is usually right-skewed, where a small number of people earn extremely high incomes.

4.These high incomes increase the mean but have little effect on the median.

5.Using the median preserves a more realistic estimate when filling in missing income values.

#Question.5-  What is forward fill and in what type of dataset is it most useful?

Answer - **Forward Fill (Forward Imputation)**

Forward fill is a method of handling missing values by replacing each missing value with the last valid (previous) observation.
It assumes that the previous value remains valid until a new value is recorded.

**Most Useful For**

Forward fill is most useful for time-series datasets, where observations are ordered by time and the previous value is a reasonable estimate until a new one becomes available.

Examples of time-series datasets:

1.Stock market prices

2.Weather data

3.Sensor or IoT readings

4.Daily inventory levels

5.Financial transaction records

#Question.6 - Why should flagging missing values be done before imputation in an ETL workflow ?

Answer -

**Preserves information**: Once missing values are replaced, you can no longer tell which values were originally missing.

**Improves transparency**: A flag identifies imputed values, making it clear which data was modified during ETL.

**Supports better analysis**: Analysts and machine learning models can use the flag to account for the fact that a value was originally missing.

**Enables auditing**: It provides a record of where imputations occurred, making the ETL process easier to review and validate.

**Helps detect data quality issues**: A high number of flagged values may indicate problems in the source system or data pipeline that need investigation.

#Question.7 -Consider a scenario where income is missing for many customers.How can this missingness itself provide business insights?

Answer - Missing income values can provide valuable business insights because the pattern of missingness may reveal customer behavior or process issues.

**Business insights from missing income data**

**1.Identify customer segments**: Customers who do not disclose income may belong to a specific demographic or privacy-conscious group.

**2.Improve marketing strategies**: Customers with missing income may require different products or personalized campaigns.

**3.Detect process issues**: A high number of missing values may indicate problems with data collection, application forms, or ETL processes.

**4.Assess customer engagement**: Customers who skip income questions may be less engaged or less willing to share personal information.

**5.Support risk analysis**: In banking or lending, missing income information may indicate applications that require additional verification before approval.

**6.Guide business decisions**: If many customers leave income blank, the company may simplify forms, make the field optional, or improve data collection methods.

#SECTION B – PRACTICAL QUESTIONS

#Question-8 Listwise Deletion
Remove all rows where Region is missing.

Tasks:
Identify affected rows

Show the dataset after deletion

Mention how many records were lose

Answer -

1. Affected rows (Region = NaN)

Customer_ID 102 - Anjali Rao

Customer_ID 105 - Amit Verma

2.Dataset after deletion
| Customer_ID | Name        | City    | Monthly_Sales | Income | Region |
| ----------- | ----------- | ------- | ------------: | -----: | ------ |
| 101         | Rahul Mehta | Mumbai  |         12000 |  65000 | West   |
| 103         | Suresh Iyer | Chennai |         15000 |    NaN | South  |
| 104         | Neha Singh  | Pune    |         18000 |  72000 | South  |
| 106         | Karan Shah  | Delhi   |         14000 |  58000 | West   |
| 107         | Pooja Das   | Kolkata |         16000 |  61000 | East   |
| 108         | Riya Kapoor | Jaipur  |           NaN |  69000 | North  |

3. Records lost

Original records: 8

Remaining records: 6

Records lost: 2

#Question.9 = . Imputation
Handle missing values in Monthly_Sales using:
Forward Fill

Tasks:
Apply forward fill

Show before vs after values

Explain why forward fill is suitable her

Answer- 1. Monthly_Sales (Before vs After Forward Fill)

| Customer_ID | Monthly_Sales (Before) | Monthly_Sales (After) |
| ----------- | ---------------------- | --------------------- |
| 101         | 12000                  | 12000                 |
| 102         | NaN                    | 12000                 |
| 103         | 15000                  | 15000                 |
| 104         | 18000                  | 18000                 |
| 105         | NaN                    | 18000                 |
| 106         | 14000                  | 14000                 |
| 107         | 16000                  | 16000                 |
| 108         | NaN                    | 16000                 |

2. Why forward fill is suitable here-

1.Monthly sales data is typically ordered sequentially, so previous values can reasonably estimate short missing gaps.

2.It maintains trend continuity in business reporting.

3.It avoids introducing artificial averages that may distort real sales behavior.

4.It is useful when missing values are assumed to be temporary reporting or logging gaps, not actual zero sales.

#Question.9 -  Flagging Missing Data
Create a flag column for missing Income.

Tasks:
Create Income_Missing_Flag (0 = present, 1 = missing)

Show updated dataset

Count how many customers have missing income

Answer - **Flagging Missing Data (Income)**

1. Create flag column
Income_Missing_Flag = 0 → Income is present
Income_Missing_Flag = 1 → Income is missing

2. Update Dataset

| Customer_ID | Name        | Income | Income_Missing_Flag |
| ----------- | ----------- | -----: | ------------------: |
| 101         | Rahul Mehta |  65000 |                   0 |
| 102         | Anjali Rao  |    NaN |                   1 |
| 103         | Suresh Iyer |    NaN |                   1 |
| 104         | Neha Singh  |  72000 |                   0 |
| 105         | Amit Verma  |  58000 |                   0 |
| 106         | Karan Shah  |  61000 |                   0 |
| 107         | Pooja Das   |    NaN |                   1 |
| 108         | Riya Kapoor |  69000 |                   0 |

3. Count of customers with missing income

Total customers with missing Income = 3

Customer IDs: 102, 103, 107

